# Étape 1 - Extraction et préparation initiale

Ce notebook documente le chargement du fichier INSERSUP et la production des deux jeux
de données utilisés ensuite pour l'EDA et la modélisation.

Le fichier brut contient plus d'un million de lignes et mélange des lignes détaillées
avec des agrégats. L'extraction est **rejouée ici de bout en bout** : elle produit le jeu
de modélisation (une ligne par formation et par promotion) et le jeu analytique (tous
niveaux d'agrégation conservés, marqués par des indicateurs).

## 1. Paramètres de lecture du fichier brut

Le séparateur est le point-virgule et le fichier est encodé en UTF-8 avec BOM. Les
valeurs `nd` et `ns` représentent des informations non disponibles ou non significatives.
La lecture par blocs est nécessaire car le fichier brut pèse 773 Mo.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_PATH = Path.cwd().parent
RAW_PATH = PROJECT_PATH / 'csv' / 'dataset.csv'
MODEL_PATH = PROJECT_PATH / 'csv' / 'dataset_phase1_modelisation.csv'
ANALYTIC_PATH = PROJECT_PATH / 'csv' / 'dataset_phase1_analytique.csv'
FUNNEL_PATH = PROJECT_PATH / 'csv' / 'entonnoir_extraction.json'

READ_PARAMS = {
    'sep': ';',
    'encoding': 'utf-8-sig',
    'dtype': str,
    'na_values': ['nd', 'ns'],
}
CHUNK_SIZE = 200_000

print(f'Fichier brut présent : {RAW_PATH.exists()}')
print(f'Jeu de modélisation déjà produit : {MODEL_PATH.exists()}')
print(f'Jeu analytique déjà produit : {ANALYTIC_PATH.exists()}')

Fichier brut présent : True
Jeu de modélisation déjà produit : True
Jeu analytique déjà produit : True


## Pourquoi ces paramètres ?

`utf-8-sig` retire le BOM présent au début du fichier. Sans ce paramètre, le nom de la
première colonne contient un caractère invisible et les sélections par nom deviennent
fragiles. `dtype=str` permet de conserver les codes et les libellés sans conversion
prématurée ; les mesures sont converties séparément après contrôle. Les valeurs `nd` et
`ns` sont traitées comme des valeurs manquantes plutôt que comme des catégories.

## 2. Origine, licence et volumétrie de la source

| Élément | Valeur |
|---|---|
| Jeu de données | INSERSUP : insertion professionnelle des diplômés de l'enseignement supérieur |
| Producteur | Ministère de l'Enseignement supérieur et de la Recherche (SIES) |
| Millésime de diffusion | `2026_S1` |
| Portail | data.enseignementsup-recherche.gouv.fr |
| Licence | Licence Ouverte / Open Licence (Etalab) : réutilisation libre avec mention de la source |
| Format | CSV, séparateur `;`, encodage UTF-8 avec BOM |
| Volume | 773 Mo, 1 036 781 lignes × 101 colonnes |
| Période couverte | Promotions 2019 à 2024 |
| Acquisition | Téléchargement manuel, fichier présent en local dans `Projet/csv/` |

**Une seule source à ce stade.** L'ajout d'une seconde source dans un second format
(API JSON du même portail, ou référentiel des établissements joint par
`Code UAI de l'établissement`) reste le point ouvert du cadrage, documenté en
[étape 0](../etapes/étape_0.md) section 4.

## 3. Contrôle du fichier brut

Avant tout traitement, on vérifie que l'en-tête se lit correctement avec les paramètres
retenus : nombre de colonnes, absence de BOM résiduel, accents corrects.

In [2]:
header = pd.read_csv(RAW_PATH, **READ_PARAMS, nrows=0)
print('Nombre de colonnes du fichier brut :', len(header.columns))
print('Première colonne :', repr(header.columns[0]))
print('Premières colonnes :', header.columns[:8].tolist())
print('Paramètres documentés :', READ_PARAMS)

Nombre de colonnes du fichier brut : 101
Première colonne : 'Diffusion des données'
Premières colonnes : ['Diffusion des données', 'Région', 'Académie', 'Établissement', 'Type de diplôme', 'Domaine disciplinaire', 'Discipline', 'Secteur disciplinaire']
Paramètres documentés : {'sep': ';', 'encoding': 'utf-8-sig', 'dtype': <class 'str'>, 'na_values': ['nd', 'ns']}


## 4. Reproduction de l'extraction

INSERSUP est un cube **avec ses marges** : les lignes de total cohabitent avec le détail
dans le même fichier. Quatre familles de totaux sont présentes.

| Famille | Colonnes concernées | Valeur de total |
|---|---|---|
| Démographique | `Genre`, `Nationalité`, `Régime d'inscription` | `ensemble` |
| Obtention | `Obtention du diplôme` | `ensemble` (vs `diplômé`) |
| Temporelle | `Promotion` | cumuls de deux années (`2019,2020`) |
| Géographique | `Région`, `Établissement` | `National` |
| Disciplinaire | `Domaine disciplinaire`, `Discipline`, `Secteur disciplinaire` | `Tous domaines...`, `Toutes disciplines`, `Tous secteurs...` |

Entraîner un modèle sur le fichier brut reviendrait à apprendre sur des lignes qui
recomptent leurs propres sous-lignes, avec fuite de la cible du total vers ses
composantes. L'extraction produit donc **deux jeux distincts** :

- le **jeu de modélisation**, où toutes les marges sont retirées : une ligne = une
  formation × une promotion simple ;
- le **jeu analytique**, où tous les niveaux sont conservés mais **étiquetés** par trois
  indicateurs booléens, ce qui permet à l'EDA de répondre aux questions sur le genre, la
  nationalité et le régime d'inscription sans jamais mélanger un total et un détail.

La cellule suivante est **protégée par une garde de cache** : le fichier brut n'est
retraité que si l'un des jeux produits est absent. Mettre `FORCE_EXTRACTION = True` pour
rejouer l'extraction complète (environ 3 à 5 minutes).

In [3]:
import json

TARGET = "6-Taux d'emploi salarié en France - 6 mois après le diplôme"

DIMENSIONS = [
    'Région', 'Académie', 'Établissement', 'Type de diplôme', 'Domaine disciplinaire',
    'Discipline', 'Secteur disciplinaire', 'Libellé du diplôme', 'Promotion',
    "Code UAI de l'établissement", 'Code du diplôme SISE',
]
MESURES = [
    '6-Nombre de sortants - 6 mois après le diplôme',
    '6-Nombre de poursuivants - 6 mois après le diplôme',
    TARGET,
    "12-Taux d'emploi salarié en France - 12 mois après le diplôme",
    "18-Taux d'emploi salarié en France - 18 mois après le diplôme",
    "24-Taux d'emploi salarié en France - 24 mois après le diplôme",
    "30-Taux d'emploi salarié en France - 30 mois après le diplôme",
    '6-Nombre de sortants en emploi non salarié - 6 mois après le diplôme',
    '6-Taux de sortants en emploi non salarié - 6 mois après le diplôme',
    '6-Nombre de sortants en emploi stable - 6 mois après le diplôme',
    '6-Taux de sortants en emploi stable - 6 mois après le diplôme',
]
COLONNES_FILTRE = ['Genre', 'Nationalité', "Régime d'inscription", 'Obtention du diplôme']
COLONNES_ANALYTIQUE_SUP = ['6-Flag - 6 mois après le diplôme',
                           '6-Exception - 6 mois après le diplôme']
USECOLS = DIMENSIONS + MESURES + COLONNES_FILTRE + COLONNES_ANALYTIQUE_SUP

MARGES = {
    'Région': 'National',
    'Établissement': 'National',
    'Domaine disciplinaire': 'Tous domaines disciplinaires',
    'Discipline': 'Toutes disciplines',
    'Secteur disciplinaire': 'Tous secteurs disciplinaires',
}

# Toute colonne demandée doit exister : on échoue bruyamment plutôt que de filtrer en silence.
manquantes = [colonne for colonne in USECOLS if colonne not in header.columns]
assert not manquantes, f'Colonnes absentes du fichier brut : {manquantes}'

FORCE_EXTRACTION = False
cache_valide = MODEL_PATH.exists() and ANALYTIC_PATH.exists() and FUNNEL_PATH.exists()

if cache_valide and not FORCE_EXTRACTION:
    entonnoir = json.loads(FUNNEL_PATH.read_text(encoding='utf-8'))
    print('Extraction déjà produite : lecture du cache.')
    print('Mettre FORCE_EXTRACTION = True pour rejouer le traitement complet.')
else:
    entonnoir = dict.fromkeys(
        ['Fichier brut', 'Cible renseignée', 'Démographie = ensemble',
         'Obtention = diplômé', 'Promotion simple', 'Hors marges géo. et disc.'], 0)
    blocs_modelisation, blocs_analytique = [], []

    for bloc in pd.read_csv(RAW_PATH, **READ_PARAMS, usecols=USECOLS, chunksize=CHUNK_SIZE):
        entonnoir['Fichier brut'] += len(bloc)

        # Le jeu analytique ne retient qu'un critère : la cible doit être mesurée.
        bloc = bloc[bloc[TARGET].notna()]
        entonnoir['Cible renseignée'] += len(bloc)

        demographie_ensemble = (bloc['Genre'].eq('ensemble')
                                & bloc['Nationalité'].eq('ensemble')
                                & bloc["Régime d'inscription"].eq('ensemble'))
        promotion_cumulee = bloc['Promotion'].str.contains(',')
        marge_geo_disc = False
        for colonne, marge in MARGES.items():
            marge_geo_disc = marge_geo_disc | bloc[colonne].eq(marge)

        analytique = bloc.copy()
        analytique['ligne_agregee'] = demographie_ensemble
        analytique['promotion_cumulee'] = promotion_cumulee
        analytique['marge_geo_disc'] = marge_geo_disc
        blocs_analytique.append(analytique)

        # Le jeu de modélisation retire ensuite chaque famille de marges.
        bloc = bloc[demographie_ensemble]
        entonnoir['Démographie = ensemble'] += len(bloc)
        bloc = bloc[bloc['Obtention du diplôme'].eq('diplômé')]
        entonnoir['Obtention = diplômé'] += len(bloc)
        bloc = bloc[~bloc['Promotion'].str.contains(',')]
        entonnoir['Promotion simple'] += len(bloc)
        for colonne, marge in MARGES.items():
            bloc = bloc[bloc[colonne].ne(marge)]
        entonnoir['Hors marges géo. et disc.'] += len(bloc)
        blocs_modelisation.append(bloc[DIMENSIONS + MESURES])

    df_modelisation = pd.concat(blocs_modelisation, ignore_index=True)
    df_analytique = pd.concat(blocs_analytique, ignore_index=True)

    # Conversion explicite : les mesures deviennent numériques, la promotion un entier.
    for cadre in (df_modelisation, df_analytique):
        for colonne in MESURES:
            cadre[colonne] = pd.to_numeric(cadre[colonne], errors='raise')
    df_modelisation['Promotion'] = df_modelisation['Promotion'].astype(int)
    df_modelisation['promotion_debut'] = df_modelisation['Promotion']
    df_analytique['promotion_debut'] = (df_analytique['Promotion']
                                        .str.slice(0, 4).astype(int))

    df_modelisation.to_csv(MODEL_PATH, index=False)
    df_analytique.to_csv(ANALYTIC_PATH, index=False)
    FUNNEL_PATH.write_text(json.dumps(entonnoir, ensure_ascii=False, indent=2),
                           encoding='utf-8')
    print('Extraction rejouée et jeux réécrits.')

Extraction déjà produite : lecture du cache.
Mettre FORCE_EXTRACTION = True pour rejouer le traitement complet.


In [4]:
entonnoir_df = pd.DataFrame({'Lignes restantes': pd.Series(entonnoir)})
entonnoir_df['Perte'] = entonnoir_df['Lignes restantes'].diff().fillna(0).astype(int)
entonnoir_df['Part du brut'] = (
    100 * entonnoir_df['Lignes restantes'] / entonnoir_df['Lignes restantes'].iloc[0]
).round(1)
print("Entonnoir de filtrage de l'extraction :")
entonnoir_df

Entonnoir de filtrage de l'extraction :


,Lignes restantes,Perte,Part du brut
Fichier brut,1036781,0,100.0
Cible renseignée,438696,-598085,42.3
Démographie = ensemble,78535,-360161,7.6
Obtention = diplômé,37472,-41063,3.6
Promotion simple,27149,-10323,2.6
Hors marges géo. et disc.,17065,-10084,1.6


## 5. Contrôle du jeu produit

On relit les fichiers écrits pour vérifier qu'ils correspondent à ce qui était attendu :
dimensions, présence de la cible, absence de doublon complet, types.

In [5]:
df = pd.read_csv(MODEL_PATH)
print(f'Dimensions du jeu de modélisation : {df.shape}')
print(f'Cible présente : {TARGET in df.columns}')
print(f'Doublons complets : {df.duplicated().sum()}')
print(f"Valeurs manquantes sur la cible : {df[TARGET].isna().sum()}")
df.head()

Dimensions du jeu de modélisation : (17065, 23)


Cible présente : True


Doublons complets : 0
Valeurs manquantes sur la cible : 0


,Région,Académie,Établissement,Type de diplôme,Domaine disciplinaire,Discipline,Secteur disciplinaire,Libellé du diplôme,Promotion,Code UAI de l'établissement,...,6-Taux d'emploi salarié en France - 6 mois après le diplôme,12-Taux d'emploi salarié en France - 12 mois après le diplôme,18-Taux d'emploi salarié en France - 18 mois après le diplôme,24-Taux d'emploi salarié en France - 24 mois après le diplôme,30-Taux d'emploi salarié en France - 30 mois après le diplôme,6-Nombre de sortants en emploi non salarié - 6 mois après le diplôme,6-Taux de sortants en emploi non salarié - 6 mois après le diplôme,6-Nombre de sortants en emploi stable - 6 mois après le diplôme,6-Taux de sortants en emploi stable - 6 mois après le diplôme,promotion_debut
0,Pays de la Loire,Nantes,École de gestion et de commerce de Vendée,Diplôme visé niveau bac + 3,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DE GESTION ET DE COMMERCE D...,2019,0851465F,...,90.91,90.91,90.91,95.45,77.27,0,0.00,6,30.00,2019
1,Auvergne-Rhône-Alpes,Lyon,EM Lyon Business School,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DE MANAGEMENT DE LYON (PGE),2020,0690197P,...,52.89,60.96,63.84,63.04,62.56,0,0.00,270,84.38,2020
2,Île-de-France,Versailles,EDC Paris Business School,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DES DIRIGEANTS ET CREATEURS...,2024,0922007G,...,45.11,58.09,51.47,NaN,NaN,0,0.00,40,66.67,2024
3,Île-de-France,Versailles,École des hautes études commerciales de Paris,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DES HAUTES ETUDES COMMERCIA...,2019,0783054W,...,35.63,32.95,35.25,36.40,36.78,0,0.00,79,84.95,2019
4,Île-de-France,Versailles,École des hautes études commerciales de Paris,Diplôme visé niveau bac + 5 grade master,"Droit, économie, gestion","Sciences économiques, gestion",Sciences de gestion,DIPLOME DE L'ECOLE DES HAUTES ETUDES COMMERCIA...,2023,0783054W,...,42.54,54.00,55.40,57.36,55.40,34,5.46,221,83.40,2023


In [6]:
resume_types = pd.DataFrame({
    'type': df.dtypes.astype(str),
    'valeurs_manquantes': df.isna().sum(),
    'modalites_distinctes': df.nunique(),
})
resume_types['part_manquante_%'] = (100 * resume_types['valeurs_manquantes'] / len(df)).round(1)

print('Colonnes catégorielles :', int((resume_types['type'] == 'str').sum()),
      '| Colonnes numériques :', int((resume_types['type'] != 'str').sum()))
print('Mémoire occupée :', round(df.memory_usage(deep=True).sum() / 1024**2, 1), 'Mo')
resume_types

Colonnes catégorielles : 10 | Colonnes numériques : 13
Mémoire occupée : 15.4 Mo


,type,valeurs_manquantes,modalites_distinctes,part_manquante_%
Région,str,0,19,0.0
Académie,str,0,32,0.0
Établissement,str,0,329,0.0
Type de diplôme,str,0,11,0.0
Domaine disciplinaire,str,0,4,0.0
Discipline,str,0,14,0.0
Secteur disciplinaire,str,0,49,0.0
Libellé du diplôme,str,0,1290,0.0
Promotion,int64,0,6,0.0
Code UAI de l'établissement,str,0,365,0.0


In [7]:
analytique_apercu = pd.read_csv(
    ANALYTIC_PATH,
    usecols=['ligne_agregee', 'promotion_cumulee', 'marge_geo_disc',
             'Genre', "Régime d'inscription"],
)
print('Jeu analytique :', analytique_apercu.shape[0], 'lignes')
print('Répartition des niveaux d\'agrégation :')
print(analytique_apercu[['ligne_agregee', 'promotion_cumulee', 'marge_geo_disc']]
      .value_counts()
      .rename('lignes')
      .to_frame())
del analytique_apercu

Jeu analytique : 438696 lignes
Répartition des niveaux d'agrégation :
                                                lignes
ligne_agregee promotion_cumulee marge_geo_disc        
False         False             False           119427
                                True            118455
              True              False            99057
True          False             False            36464
False         True              True             23222
True          False             True             20701
              True              False            19140
                                True              2230


## 6. Seconde source : le référentiel des établissements (API JSON)

Le cadrage de l'étape 0 laissait un manque explicite : une seule source, un seul format.
Il est comblé ici par le **référentiel des principaux établissements d'enseignement
supérieur**, publié par le même ministère mais constitué indépendamment d'InserSup, et
exposé en **JSON** par l'API Explore v2.1 d'Opendatasoft.

| Élément | Valeur |
|---|---|
| Jeu | `fr-esr-principaux-etablissements-enseignement-superieur` |
| Format | JSON via API REST, contre CSV séparé par `;` pour la source principale |
| Point d'accès | `data.enseignementsup-recherche.gouv.fr/api/explore/v2.1/.../exports/json` |
| Licence | Licence Ouverte / Open Licence (Etalab) |
| Clé de jointure | `uai` vers `Code UAI de l'établissement` |

L'intérêt ne se limite pas à cocher un critère de format. Ce référentiel porte des
variables qu'InserSup ne contient pas : le **secteur public ou privé** de l'établissement,
sa **typologie**, son **statut juridique** et son **effectif d'inscrits**.

In [8]:
import urllib.request

REFERENTIEL_URL = (
    'https://data.enseignementsup-recherche.gouv.fr/api/explore/v2.1/catalog/'
    'datasets/fr-esr-principaux-etablissements-enseignement-superieur/exports/json')
REFERENTIEL_PATH = PROJECT_PATH / 'csv' / 'referentiel_etablissements.json'

# Le fichier est conservé en local après le premier appel : le notebook reste exécutable
# hors ligne, et la source se rejoue en supprimant le fichier.
if not REFERENTIEL_PATH.exists():
    print('Téléchargement depuis l\'API...')
    urllib.request.urlretrieve(REFERENTIEL_URL, REFERENTIEL_PATH)

with open(REFERENTIEL_PATH, encoding='utf-8') as fichier:
    referentiel = pd.json_normalize(json.load(fichier))

print(f'Format lu : JSON · {REFERENTIEL_PATH.stat().st_size / 1024:.0f} Ko')
print(f'Enregistrements : {len(referentiel)} · champs : {referentiel.shape[1]}')
print(f'Codes UAI renseignés : {referentiel["uai"].notna().sum()} · '
      f'distincts : {referentiel["uai"].nunique()}')
assert referentiel['uai'].is_unique, 'La clé de jointure n\'est pas unique'

# Les champs retenus sont ceux qu'InserSup ne fournit pas. `type_d_etablissement` arrive
# en liste JSON : c'est la différence de structure qu'un CSV plat ne peut pas porter.
CHAMPS_REFERENTIEL = {
    'uai': "Code UAI de l'établissement",
    'secteur_d_etablissement': 'secteur_etablissement',
    'statut_juridique_court': 'statut_juridique_etablissement',
    'dep_nom': 'departement_etablissement',
    'inscrits_2022': 'inscrits_etablissement',
}
etablissements = referentiel[list(CHAMPS_REFERENTIEL)].rename(columns=CHAMPS_REFERENTIEL)
etablissements['type_etablissement'] = referentiel['type_d_etablissement'].apply(
    lambda valeur: valeur[0] if isinstance(valeur, list) and valeur else None)

etablissements.head()

Format lu : JSON · 805 Ko
Enregistrements : 245 · champs : 100
Codes UAI renseignés : 245 · distincts : 245


,Code UAI de l'établissement,secteur_etablissement,statut_juridique_etablissement,departement_etablissement,inscrits_etablissement,type_etablissement
0,0840685N,public,EPSCP,Vaucluse,6346.0,Université
1,0942283W,privé,Association loi de 1901,Val-de-Marne,NaN,École
2,0670189S,public,EPSCP,Bas-Rhin,NaN,École
3,0860073M,public,EPSCP,Vienne,768.0,École
4,0290124C,public,EPSCP,Finistère,NaN,Grand établissement


In [9]:
# Jointure sur le jeu de modélisation produit plus haut, en gauche : aucune ligne
# d'InserSup ne doit disparaître parce qu'un établissement manque au référentiel.
avant = len(df)
enrichi = df.merge(etablissements, on="Code UAI de l'établissement", how='left')
assert len(enrichi) == avant, 'La jointure a dupliqué des lignes'

couvert = enrichi['secteur_etablissement'].notna()
uai_total = df["Code UAI de l'établissement"].nunique()
uai_couverts = df.loc[couvert.to_numpy(), "Code UAI de l'établissement"].nunique()

print(f'Lignes après jointure : {len(enrichi)} (inchangé)')
print(f'Établissements appariés : {uai_couverts} / {uai_total} '
      f'({100 * uai_couverts / uai_total:.1f} %)')
print(f'Lignes couvertes       : {int(couvert.sum())} / {len(enrichi)} '
      f'({100 * couvert.mean():.1f} %)')
print()

apport = enrichi.loc[couvert, ['secteur_etablissement', TARGET]].groupby(
    'secteur_etablissement')[TARGET].agg(
        **{'formations': 'size', 'taux d\'emploi moyen': 'mean',
           'médiane': 'median', 'écart-type': 'std'}).round(2)
print("Ce que la seconde source ajoute : un clivage invisible dans InserSup")
apport

Lignes après jointure : 17065 (inchangé)
Établissements appariés : 184 / 365 (50.4 %)
Lignes couvertes       : 13604 / 17065 (79.7 %)

Ce que la seconde source ajoute : un clivage invisible dans InserSup


,formations,taux d'emploi moyen,médiane,écart-type
secteur_etablissement,,,,
privé,849,51.67,54.78,20.85
public,12755,56.21,56.25,18.31


**Couverture partielle, et ce n'est pas un défaut de jointure.** Le référentiel ne
recense que les 245 établissements *principaux* : universités, écoles d'ingénieurs, grands
établissements. InserSup descend, lui, jusqu'aux écoles privées de petite taille. La
moitié des codes UAI reste donc sans correspondance, mais les établissements absents sont
les plus petits : les lignes appariées représentent **79,7 % du jeu** pour 50,4 % des
établissements.

**Le clivage public / privé est réel** : 56,21 % de taux d'emploi moyen dans le public
contre 51,67 % dans le privé, soit **4,5 points d'écart**, avec une dispersion plus forte
côté privé (écart-type 20,9 contre 18,3). L'information n'existe nulle part dans InserSup
et ne se déduit d'aucune colonne déjà présente : c'est ce qu'on attend d'un enrichissement.
La comparaison reste déséquilibrée (849 formations privées appariées contre 12 755
publiques) et le privé est justement la partie la moins bien couverte : l'écart est un
signal à creuser à l'étape 2, pas une conclusion.

**Ce qui n'est pas fait, et pourquoi.** Ces variables ne sont pas versées dans le modèle
des étapes 4 et 5. Avec 20 % de lignes non appariées, les faire entrer dans `X` imposerait
soit une catégorie « non référencé » qui encoderait surtout la petite taille de
l'établissement, soit une imputation qui inventerait une information administrative. Les
deux relèvent d'une décision de modélisation à mesurer contre le modèle actuel, pas d'un
ajout en fin d'étape 1. La source est chargée, contrôlée et jointe ; son intégration au
modèle est la question suivante, pas un acquis.

## 7. Résultat de l'extraction

| Fichier | Lignes | Colonnes | Usage |
|---|---:|---:|---|
| `csv/dataset_phase1_modelisation.csv` | 17 065 | 23 | Étapes 3 à 5 : une ligne = une formation × promotion, sans double comptage |
| `csv/dataset_phase1_analytique.csv` | 438 696 | 32 | Étape 2 : tous les niveaux conservés, étiquetés par `ligne_agregee`, `promotion_cumulee`, `marge_geo_disc` |

### Pourquoi `Code du diplôme SISE` est extrait

Les libellés ne sont pas des identifiants. 1 290 libellés de diplôme distincts
correspondent à 1 408 codes SISE, et 329 noms d'établissement à 365 codes UAI : des
formations différentes portent le même nom. Une clé construite sur les libellés laisse
**107 lignes en conflit** ; la clé `Code UAI × Code SISE × Promotion` en laisse **zéro**.
Les deux codes sont donc extraits, et la vérification est faite à l'étape 2.

Le jeu analytique est exclu du dépôt git par `.gitignore` : il dépasse la limite de
100 Mo par fichier de GitHub et se régénère par une exécution de ce notebook.

### Décision de grain analytique

Le fichier brut contient des marges et des lignes cumulant plusieurs promotions. Pour
éviter de comparer des niveaux de détail différents, le jeu de modélisation retient les
observations correspondant à une formation et une promotion simple. Cette décision réduit
le volume de 1 036 781 à 17 065 lignes, mais elle est ce qui rend la cible comparable
d'une ligne à l'autre et supprime le double comptage.

### Limites héritées de la source

1. **Couverture de la cible** : 42,3 % des lignes seulement portent un taux d'emploi
   salarié à 6 mois. Les autres mesures existent mais ne sont pas diffusées.
2. **Seuil de publication** : INSERSUP ne publie aucun taux sous 20 sortants. L'effectif
   minimal du jeu de modélisation est donc exactement 20, et les petites formations sont
   structurellement absentes.
3. **Colonnes vides** : neuf colonnes du fichier brut ne contiennent aucune valeur dans ce
   millésime, dont le taux d'emploi global qui était la cible envisagée au cadrage et les
   trois colonnes de salaire à 6 mois.

## Utilisation de l'IA sur cette étape

| Prompt utilisé | Ce que l'IA a produit | Vérification effectuée |
|---|---|---|
| « Ce CSV de 773 Mo a un BOM et des codes `nd`/`ns` : quels paramètres `read_csv` ? » | `encoding='utf-8-sig'`, `na_values`, lecture par blocs | Comparaison du nom de la première colonne avec et sans `utf-8-sig` |
| « Comment détecter les lignes de total dans un fichier statistique qui mélange marges et détail ? » | Piste des modalités `ensemble` / `National` / `Tous ...` | Recensement exhaustif des modalités de chaque colonne de dimension avant d'écrire les filtres |
| « Ma cible est vide sur 1 M de lignes, comment choisir une cible de remplacement ? » | Mesurer le taux de remplissage réel de chaque colonne candidate | Remplissage recalculé colonne par colonne, choix tranché sur les 42,3 % de `6-Taux d'emploi salarié en France` |

Le code d'extraction a été relu ligne à ligne : la garde de cache et l'ordre des filtres
de l'entonnoir ont été corrigés à la main, l'IA proposant initialement de charger le
fichier entier en mémoire.